# STRATA, perfil do kernel D na GPU (sem modelo)

**Antes de rodar:** *Ambiente de execução → Alterar o tipo de ambiente de execução* → **A100**. Depois *Executar tudo*. Leva ~20 min, quase tudo compilando o ggml com CUDA; não baixa modelo.

No Llama-3.1-8B (`docs/11`, resultado), a fusão gate+up deu +2,7% ao P e +3,1% ao D exato no nível 1, e **nada** ao D linha (135,9 → 136,1 tok/s). Esses ~3% são quase tudo o que falta para o D linha passar de 0,92 a 0,95 do Q5_K_M. Este notebook isola o kernel:

1. **Tempo por matriz**, nas formas do FFN do 8B (4096 → 14336, uma coluna): só o up, e gate + up + swiglu (fundidos num kernel), para cada tipo, com e sem fusão. Se a razão "gate+up / só up" do D1 linha ficar perto de 2 e a do P1 bem abaixo, a fusão não funciona para ele.
2. **Nsight Compute** (`ncu`), se o Colab deixar ler os contadores: registradores, ocupação, instruções e banda de memória dos kernels do P1, D1 e D1 linha, fundidos e não.

In [ ]:
import os, re, io, subprocess, shutil
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap,driver_version', '--format=csv,noheader'], capture_output=True, text=True)
assert gpu.returncode == 0, 'Sem GPU NVIDIA: troque o tipo de ambiente de execução para A100.'
GPU = gpu.stdout.strip().splitlines()[0]
print('GPU:', GPU)
if 'A100' not in GPU:
    print('Aviso: o resultado do 8B foi no A100; em outra GPU o perfil pode ser outro.')

In [ ]:
STRATA_SHA = "0270529005a105c14291c85ccfa558f91f412b4f"
LLAMA_SHA = "2a667032dc788c3a071d538e0b572e45a3f12731"
%cd /content
def clona(pasta, url, sha):
    !rm -rf {pasta} && git init -q {pasta} && cd {pasta} && git remote add origin {url} && git fetch -q --depth 1 origin {sha} && git checkout -q FETCH_HEAD
clona('strata', 'https://github.com/Cantyll/strata.git', STRATA_SHA)
clona('llama.cpp', 'https://github.com/ggml-org/llama.cpp.git', LLAMA_SHA)
!sh strata/kernel/ggml/aplicar.sh llama.cpp
!cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native -DGGML_NATIVE=ON 2>&1 | tail -2 && cmake --build build -j $(nproc) --target ggml ggml-base ggml-cpu ggml-cuda 2>&1 | tail -5
LB = '/content/llama.cpp/build/bin'
for p in ['strata_cuda_check', 'strata_cuda_bench']:
    !cc -O2 strata/tests/kernel/{p}.c -I llama.cpp/ggml/include -L {LB} -lggml -lggml-base -lggml-cpu -lggml-cuda -lm -Wl,-rpath,{LB} -o {p}
out = !./strata_cuda_check
print(out[-1])
assert out[-1].strip() == 'tudo OK', '\n'.join(l for l in out if 'DIVERGE' in l or 'ASSERT' in l)

Tempo por matriz: 16 camadas independentes num grafo, 50 rodadas. "razão" = gate+up+swiglu / só o up (2 = a fusão não economiza nada).

In [ ]:
com = !./strata_cuda_bench 16 50
sem = !GGML_CUDA_DISABLE_FUSION=1 ./strata_cuda_bench 16 50
print('\n'.join(com)); print(); print('\n'.join(sem))
if len(com) < 3 or len(sem) < 3:
    print('O strata_cuda_bench parou antes de terminar: cole a saída acima na thread.')
def tabela(linhas):
    t = {}
    for l in linhas:
        c = l.split()
        if len(c) == 4 and re.match(r'^[0-9.]+$', c[1]):
            t[c[0]] = (float(c[1]), float(c[2]))
    return t
TC, TS = tabela(com), tabela(sem)

Nsight Compute nos kernels do MMVQ do P1, D1 e D1 linha (2 camadas, 3 rodadas por caso). Se aparecer `ERR_NVGPUCTRPERM`, o Colab não deixou ler os contadores: o resto do notebook vale assim mesmo.

In [ ]:
NCU = shutil.which('ncu') or next((p for p in ['/usr/local/cuda/bin/ncu', '/opt/nvidia/nsight-compute/ncu'] if os.path.exists(p)), None)
if not NCU:
    !ls /usr/local/cuda/bin | head -50
    !ls /opt/nvidia 2>/dev/null
print('ncu:', NCU)
MET = ','.join(['gpu__time_duration.sum', 'launch__registers_per_thread', 'launch__block_size', 'launch__grid_size',
                'launch__occupancy_limit_registers', 'sm__warps_active.avg.pct_of_peak_sustained_active',
                'smsp__inst_executed.sum', 'smsp__issue_active.avg.pct_of_peak_sustained_active',
                'dram__throughput.avg.pct_of_peak_sustained_elapsed', 'sm__throughput.avg.pct_of_peak_sustained_elapsed'])
perfil = {}
if NCU:
    import pandas as pd
    for caso in ['q5_K', 'p1', 'd1', 'd1_linha']:
        r = subprocess.run(f'{NCU} --kernel-name regex:mul_mat_vec_q --print-kernel-base mangled --metrics {MET} --csv --page raw ./strata_cuda_bench 2 3 {caso}',
                           shell=True, capture_output=True, text=True)
        txt = r.stdout[r.stdout.find('"ID"'):] if '"ID"' in r.stdout else ''
        if not txt:
            print(caso, 'sem perfil:', (r.stdout + r.stderr)[-600:])
            continue
        df = pd.read_csv(io.StringIO(txt)).iloc[1:]   # a 2ª linha traz as unidades
        # nome mangled: _Z13mul_mat_vec_qIL9ggml_type<tipo>ELi<colunas>ELb<fusão>E...; tipo 57 a 60 = kernel linha
        m = df['Kernel Name'].str.extract(r'mul_mat_vec_qIL9ggml_type(\d+)ELi\d+ELb([01])E')
        df['tipo'], df['fundido'] = m[0], m[1] == '1'
        if df['tipo'].isna().all():
            print(caso, 'nomes de kernel inesperados:', df['Kernel Name'].unique()[:4])
        for c in MET.split(','):
            df[c] = pd.to_numeric(df[c].astype(str).str.replace(',', ''), errors='coerce')
        perfil[caso] = df.groupby(['tipo', 'fundido'])[MET.split(',')].median()
        print(caso); display(perfil[caso].T)

Resumo para colar na thread.

In [ ]:
T = [f'# STRATA, perfil do kernel D (MMVQ, uma coluna, FFN 4096 → 14336)', '', f'GPU: {GPU}. STRATA `{STRATA_SHA[:7]}`.', '',
     '| tipo | só o up (µs) | gate+up (µs) | razão | gate+up sem fusão (µs) | ganho da fusão |', '|---|---:|---:|---:|---:|---:|']
for k, (u, f) in TC.items():
    s = TS.get(k, (float('nan'), float('nan')))[1]
    T.append(f'| {k} | {u:.2f} | {f:.2f} | {f / u:.3f} | {s:.2f} | {s / f - 1:+.1%} |')
for caso, df in perfil.items():
    T += ['', f'## ncu: {caso}', '', df.T.to_markdown()]
open('/content/strata_kernel_d_perfil.md', 'w').write('\n'.join(T) + '\n')
from IPython.display import Markdown, display
display(Markdown('\n'.join(T)))
from google.colab import files
files.download('/content/strata_kernel_d_perfil.md')